In [1]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold, cross_val_score
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import sklearn.model_selection as skmodel
from sklearn.ensemble import RandomForestClassifier
import sklearn.metrics as skmet
import sklearn.neighbors as skng
from sklearn.metrics import accuracy_score
import joblib

In [2]:
df = pd.read_csv("train.csv")
df

,User_ID,Age,Gender,Platform,Daily_Usage_Time (minutes),Posts_Per_Day,Likes_Received_Per_Day,Comments_Received_Per_Day,Messages_Sent_Per_Day,Dominant_Emotion
0,1,25,Female,Instagram,120.0,3.0,45.0,10.0,12.0,Happiness
1,2,30,Male,Twitter,90.0,5.0,20.0,25.0,30.0,Anger
2,3,22,Male,Facebook,60.0,2.0,15.0,5.0,20.0,Neutral
3,4,28,Female,Instagram,200.0,8.0,100.0,30.0,50.0,Anxiety
4,5,33,Male,LinkedIn,45.0,1.0,5.0,2.0,10.0,Boredom
...,...,...,...,...,...,...,...,...,...,...
996,996,33,Female,Twitter,85.0,4.0,35.0,18.0,18.0,Boredom
997,997,22,Female,Facebook,70.0,1.0,14.0,6.0,10.0,Neutral
998,998,35,Male,Whatsapp,110.0,3.0,50.0,25.0,25.0,Happiness
999,999,28,Female,Telegram,60.0,2.0,18.0,8.0,18.0,Anger


In [3]:
df_test = pd.read_csv("test.csv")
df_test

,User_ID,Age,Gender,Platform,Daily_Usage_Time (minutes),Posts_Per_Day,Likes_Received_Per_Day,Comments_Received_Per_Day,Messages_Sent_Per_Day,Dominant_Emotion
0,500,27,Female,Snapchat,120,4,40,18,22,Neutral
1,488,21,Male,Snapchat,60,1,18,7,12,Neutral
2,776,28,Male,Snapchat,115,3,38,18,27,Anxiety
3,869,27,Male,Telegram,105,3,48,20,28,Anxiety
4,573,21,Female,Facebook,55,3,17,7,12,Neutral
...,...,...,...,...,...,...,...,...,...,...
96,184,22,Male,Twitter,70,1,20,5,22,Neutral
97,896,33,Male,LinkedIn,85,4,35,18,18,Boredom
98,708,24,Male,Facebook,110,6,35,23,25,Sadness
99,993,24,Male,Telegram,75,3,37,16,22,Neutral


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   User_ID                     1001 non-null   object 
 1   Age                         1001 non-null   object 
 2   Gender                      1000 non-null   object 
 3   Platform                    1000 non-null   object 
 4   Daily_Usage_Time (minutes)  1000 non-null   float64
 5   Posts_Per_Day               1000 non-null   float64
 6   Likes_Received_Per_Day      1000 non-null   float64
 7   Comments_Received_Per_Day   1000 non-null   float64
 8   Messages_Sent_Per_Day       1000 non-null   float64
 9   Dominant_Emotion            1000 non-null   object 
dtypes: float64(5), object(5)
memory usage: 78.3+ KB


In [5]:
df["Age"].unique()

array(['25', '30', '22', '28', '33', '21', '27', '24', '29', '31', '23',
       '26', '34', '35', '32', 'Male', 'Female',
       ' işte mevcut veri kümesini 1000 satıra tamamlıyorum:'],
      dtype=object)

In [6]:
cn= "Age"
v = " işte mevcut veri kümesini 1000 satıra tamamlıyorum:"

intodr = df[df[cn]==v].index
df =df.drop(intodr)
df["Age"].value_counts()

Age
27        92
28        92
29        90
22        74
26        66
24        64
25        64
31        62
21        56
33        56
30        48
23        48
Male      40
35        38
32        38
34        36
Female    36
Name: count, dtype: int64

In [7]:
def swap_incorrect_values(row):
    if isinstance(row['Age'], str) and isinstance(row['Gender'], int):
        row['Age'], row['Gender'] = row['Gender'], row['Age']
    return row

In [8]:
df.apply(swap_incorrect_values, axis=1)

,User_ID,Age,Gender,Platform,Daily_Usage_Time (minutes),Posts_Per_Day,Likes_Received_Per_Day,Comments_Received_Per_Day,Messages_Sent_Per_Day,Dominant_Emotion
0,1,25,Female,Instagram,120.0,3.0,45.0,10.0,12.0,Happiness
1,2,30,Male,Twitter,90.0,5.0,20.0,25.0,30.0,Anger
2,3,22,Male,Facebook,60.0,2.0,15.0,5.0,20.0,Neutral
3,4,28,Female,Instagram,200.0,8.0,100.0,30.0,50.0,Anxiety
4,5,33,Male,LinkedIn,45.0,1.0,5.0,2.0,10.0,Boredom
...,...,...,...,...,...,...,...,...,...,...
996,996,33,Female,Twitter,85.0,4.0,35.0,18.0,18.0,Boredom
997,997,22,Female,Facebook,70.0,1.0,14.0,6.0,10.0,Neutral
998,998,35,Male,Whatsapp,110.0,3.0,50.0,25.0,25.0,Happiness
999,999,28,Female,Telegram,60.0,2.0,18.0,8.0,18.0,Anger


In [9]:
df['Dominant_Emotion'].value_counts()

Dominant_Emotion
Happiness    200
Neutral      200
Anxiety      170
Sadness      160
Boredom      140
Anger        130
Name: count, dtype: int64

In [10]:
expected_genders = ['Male', 'Female']
expected_platforms = ['Instagram', 'Twitter', 'Facebook', 'LinkedIn', 'Snapchat', 'Telegram']

In [11]:
df = df[df['Gender'].isin(expected_genders)]
df_test = df_test[df_test['Gender'].isin(expected_genders)]
df = df[df['Platform'].isin(expected_platforms)]
df_test = df_test[df_test['Platform'].isin(expected_platforms)]

In [12]:
df['Age'] = pd.to_numeric(df['Age'], errors='coerce').fillna(0).astype(int)
df['Daily_Usage_Time (minutes)'] = pd.to_numeric(df['Daily_Usage_Time (minutes)'], errors='coerce').fillna(0).astype(int)
df['Posts_Per_Day'] = pd.to_numeric(df['Posts_Per_Day'], errors='coerce').fillna(0).astype(int)
df['Likes_Received_Per_Day'] = pd.to_numeric(df['Likes_Received_Per_Day'], errors='coerce').fillna(0).astype(int)
df['Comments_Received_Per_Day'] = pd.to_numeric(df['Comments_Received_Per_Day'], errors='coerce').fillna(0).astype(int)
df['Messages_Sent_Per_Day'] = pd.to_numeric(df['Messages_Sent_Per_Day'], errors='coerce').fillna(0).astype(int)

In [13]:
combined = pd.concat([df, df_test])

In [14]:
label_encoder_platform = LabelEncoder()
label_encoder_gender = LabelEncoder()
label_encoder_emotion = LabelEncoder()

In [15]:
combined["Platform"] = label_encoder_platform.fit_transform(combined["Platform"])
combined["Gender"] = label_encoder_gender.fit_transform(combined["Gender"])
combined["Dominant_Emotion"] = label_encoder_emotion.fit_transform(combined["Dominant_Emotion"])

In [16]:
df["Platform"] = combined["Platform"].iloc[:len(df)]
df["Gender"] = combined["Gender"].iloc[:len(df)]
df["Dominant_Emotion"] = combined["Dominant_Emotion"].iloc[:len(df)]

In [17]:
df_test["Platform"] = combined["Platform"].iloc[len(df):]
df_test["Gender"] = combined["Gender"].iloc[len(df):]
df_test["Dominant_Emotion"] = combined["Dominant_Emotion"].iloc[len(df):]

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 856 entries, 0 to 1000
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   User_ID                     856 non-null    object
 1   Age                         856 non-null    int32 
 2   Gender                      856 non-null    int32 
 3   Platform                    856 non-null    int32 
 4   Daily_Usage_Time (minutes)  856 non-null    int32 
 5   Posts_Per_Day               856 non-null    int32 
 6   Likes_Received_Per_Day      856 non-null    int32 
 7   Comments_Received_Per_Day   856 non-null    int32 
 8   Messages_Sent_Per_Day       856 non-null    int32 
 9   Dominant_Emotion            856 non-null    int32 
dtypes: int32(9), object(1)
memory usage: 43.5+ KB


In [19]:
df

,User_ID,Age,Gender,Platform,Daily_Usage_Time (minutes),Posts_Per_Day,Likes_Received_Per_Day,Comments_Received_Per_Day,Messages_Sent_Per_Day,Dominant_Emotion
0,1,25,0,1,120,3,45,10,12,3
1,2,30,1,5,90,5,20,25,30,0
2,3,22,1,0,60,2,15,5,20,4
3,4,28,0,1,200,8,100,30,50,1
4,5,33,1,2,45,1,5,2,10,2
...,...,...,...,...,...,...,...,...,...,...
995,995,26,1,1,150,5,70,25,30,1
996,996,33,0,5,85,4,35,18,18,2
997,997,22,0,0,70,1,14,6,10,4
999,999,28,0,4,60,2,18,8,18,0


In [20]:
df_test

,User_ID,Age,Gender,Platform,Daily_Usage_Time (minutes),Posts_Per_Day,Likes_Received_Per_Day,Comments_Received_Per_Day,Messages_Sent_Per_Day,Dominant_Emotion
0,500,27,0,3,120,4,40,18,22,4
1,488,21,1,3,60,1,18,7,12,4
2,776,28,1,3,115,3,38,18,27,1
3,869,27,1,4,105,3,48,20,28,1
4,573,21,0,0,55,3,17,7,12,4
...,...,...,...,...,...,...,...,...,...,...
96,184,22,1,5,70,1,20,5,22,4
97,896,33,1,2,85,4,35,18,18,2
98,708,24,1,0,110,6,35,23,25,5
99,993,24,1,4,75,3,37,16,22,4


In [21]:
x = df[["Age", "Platform", "Comments_Received_Per_Day", "Likes_Received_Per_Day", "Gender", "Messages_Sent_Per_Day", "Posts_Per_Day", "Daily_Usage_Time (minutes)"]]
x_test2 = df_test[["Age", "Platform", "Comments_Received_Per_Day", "Likes_Received_Per_Day", "Gender", "Messages_Sent_Per_Day", "Posts_Per_Day", "Daily_Usage_Time (minutes)"]]
y = df["Dominant_Emotion"]

In [22]:
x

,Age,Platform,Comments_Received_Per_Day,Likes_Received_Per_Day,Gender,Messages_Sent_Per_Day,Posts_Per_Day,Daily_Usage_Time (minutes)
0,25,1,10,45,0,12,3,120
1,30,5,25,20,1,30,5,90
2,22,0,5,15,1,20,2,60
3,28,1,30,100,0,50,8,200
4,33,2,2,5,1,10,1,45
...,...,...,...,...,...,...,...,...
995,26,1,25,70,1,30,5,150
996,33,5,18,35,0,18,4,85
997,22,0,6,14,0,10,1,70
999,28,4,8,18,0,18,2,60


In [23]:
x_test2

,Age,Platform,Comments_Received_Per_Day,Likes_Received_Per_Day,Gender,Messages_Sent_Per_Day,Posts_Per_Day,Daily_Usage_Time (minutes)
0,27,3,18,40,0,22,4,120
1,21,3,7,18,1,12,1,60
2,28,3,18,38,1,27,3,115
3,27,4,20,48,1,28,3,105
4,21,0,7,17,0,12,3,55
...,...,...,...,...,...,...,...,...
96,22,5,5,20,1,22,1,70
97,33,2,18,35,1,18,4,85
98,24,0,23,35,1,25,6,110
99,24,4,16,37,1,22,3,75


In [24]:
y

0       3
1       0
2       4
3       1
4       2
       ..
995     1
996     2
997     4
999     0
1000    4
Name: Dominant_Emotion, Length: 856, dtype: int32

In [30]:
clf = DecisionTreeClassifier(random_state=42)
k_folds = KFold(n_splits = 5)
scores = cross_val_score(clf, x, y, cv = k_folds)
print("Cross Validation Scores: ", scores)
print("Average CV Score: ", scores.mean())
print("Number of CV Scores used in Average: ", len(scores))

Cross Validation Scores:  [0.8255814 1.        1.        1.        1.       ]
Average CV Score:  0.9651162790697676
Number of CV Scores used in Average:  5


In [33]:
clf2 = DecisionTreeClassifier(random_state=42)
clf2.fit(x,y)
res = clf2.predict(x_test2)
print(skmet.classification_report (df_test["Dominant_Emotion"] , res ))

              precision    recall  f1-score   support

           0       0.86      1.00      0.92         6
           1       1.00      0.90      0.95        21
           2       1.00      0.88      0.93        16
           3       0.93      1.00      0.96        13
           4       0.93      1.00      0.96        27
           5       0.86      0.86      0.86        14

    accuracy                           0.94        97
   macro avg       0.93      0.94      0.93        97
weighted avg       0.94      0.94      0.94        97



In [41]:
classifier = RandomForestClassifier(n_estimators = 100)
classifier.fit(x,y)
y_predict= classifier.predict(x_test2)
print(skmet.classification_report (df_test["Dominant_Emotion"] , y_predict))
print(accuracy_score(df_test["Dominant_Emotion"] , y_predict))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         6
           1       1.00      0.95      0.98        21
           2       1.00      0.94      0.97        16
           3       0.93      1.00      0.96        13
           4       1.00      1.00      1.00        27
           5       0.93      1.00      0.97        14

    accuracy                           0.98        97
   macro avg       0.98      0.98      0.98        97
weighted avg       0.98      0.98      0.98        97

0.979381443298969


In [42]:
classifier2 = skng.KNeighborsClassifier(n_neighbors=5)
classifier2.fit(x,y)
y_predict2= classifier2.predict(x_test2)
print(skmet.classification_report (df_test["Dominant_Emotion"] , y_predict2))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         6
           1       0.95      0.95      0.95        21
           2       0.88      0.94      0.91        16
           3       0.92      0.92      0.92        13
           4       1.00      0.93      0.96        27
           5       0.87      0.93      0.90        14

    accuracy                           0.94        97
   macro avg       0.94      0.94      0.94        97
weighted avg       0.94      0.94      0.94        97



In [43]:
joblib.dump(classifier, 'classifier.joblib')
joblib.dump(label_encoder_platform, 'label_encoder_platform.joblib')
joblib.dump(label_encoder_gender, 'label_encoder_gender.joblib')
joblib.dump(label_encoder_emotion, 'label_encoder_emotion.joblib')

['label_encoder_emotion.joblib']